# Práctica de consolidación · De la ficha de la fuente a la deformación esperada

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/modelacion-sismica-tsunami-espam/blob/main/cuadernos/Practica_consolidacion_Okada_IRD505.ipynb)


**Modelación Sísmica y Tsunami (IRD-505) · Ingeniería en Riesgos de Desastres · ESPAM MFL**
Docente: Cristian Paliz Acosta, MSc.

En la Unidad 1 cada grupo construyó la **ficha de la fuente** de su sismo. En esta práctica esa tabla se convierte en una predicción física: cuánto sube y cuánto baja el terreno y el fondo marino, dónde, y qué significa eso para la localidad costera que su grupo adoptó.

**Ruta de la práctica**

| Paso | Qué hacen | Producto |
|---|---|---|
| 0 | Predicen a mano, **antes** de calcular | Predicción registrada |
| 1 | Cargan su ficha | Ficha en el cuaderno |
| 2 | Auditan la ficha | Tabla de verificaciones |
| 3 | Verifican el código de Okada | Comparación con Okada (1985) |
| 4 | Calculan el campo de deformación | Mapa de Uz y vectores horizontales |
| 5 | Trazan el perfil por su localidad | Perfil Uz y corte de la falla |
| 6 | Comparan predicción y modelo | Tabla predicción vs. modelo |
| 7 | Validan con observaciones | Residuos y gráfico 1:1 |
| 8 | Analizan la sensibilidad | Gráfico de tornado |
| 9 | Exportan la condición inicial para la Unidad 3 | CSV, figuras y ficha final |

**Cómo usar el cuaderno.** Hagan una copia (*Archivo → Guardar una copia en Drive*). Solo editen las celdas marcadas con ✏️. Ejecuten las celdas en orden (*Entorno de ejecución → Ejecutar todas* después de editar la ficha).

## Paso 0 ✏️ · Predicción antes de calcular

Sin ejecutar nada más, con su ficha y lo visto en clase, respondan en la celda siguiente. **No se corrige la predicción: se evalúa que exista y que esté justificada.** Al final compararán con el modelo.

Pistas para razonar: ¿hacia dónde buza la falla? ¿El borde superior está bajo el mar o bajo tierra? ¿Su localidad queda sobre la falla, más allá del borde inferior o lejos?

In [ ]:
# ✏️ Completen ANTES de ejecutar el resto del cuaderno
PREDICCION = dict(
    levantamiento_max_m = 1.0,      # levantamiento máximo esperado (m), positivo
    hundimiento_max_m   = -0.3,     # hundimiento máximo esperado (m), negativo
    signo_en_localidad  = "baja",   # "sube" o "baja": ¿qué le pasa al terreno en su localidad?
    justificacion = "Escriban aquí, en dos o tres líneas, por qué esperan esos valores.",
)

## Preparación del entorno
Importa librerías y define utilidades. No hace falta editar.

In [ ]:
import json, math, warnings, urllib.request, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

KM_POR_GRADO = 111.195

# Localidades costeras adoptadas por los grupos (coordenadas aproximadas del centro poblado)
LOCALIDADES = {
    "Esmeraldas":        (0.968, -79.652),
    "Atacames":          (0.867, -79.847),
    "Muisne":            (0.610, -80.022),
    "Pedernales":        (0.071, -80.053),
    "Jama":              (-0.200, -80.263),
    "Bahía de Caráquez": (-0.600, -80.425),
    "Manta":             (-0.950, -80.716),
    "Puerto López":      (-1.560, -80.810),

}

def a_km(lat, lon, lat0, lon0):
    # Proyección local equirectangular (adecuada para unos cientos de km)
    e = (np.asarray(lon) - lon0) * KM_POR_GRADO * np.cos(np.radians(lat0))
    n = (np.asarray(lat) - lat0) * KM_POR_GRADO
    return e, n

def a_grados(e, n, lat0, lon0):
    lat = lat0 + np.asarray(n) / KM_POR_GRADO
    lon = lon0 + np.asarray(e) / (KM_POR_GRADO * np.cos(np.radians(lat0)))
    return lat, lon

def cargar_costa(bbox, archivo="ne_10m_coastline.geojson"):
    # Costa de Natural Earth (dominio público). Si no hay internet, el mapa sale sin costa.
    url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_coastline.geojson"
    try:
        if not os.path.exists(archivo):
            urllib.request.urlretrieve(url, archivo)
        with open(archivo) as f:
            gj = json.load(f)
    except Exception as err:
        print("No se pudo descargar la línea de costa; se continúa sin ella.", err)
        return []
    lon_min, lon_max, lat_min, lat_max = bbox
    tramos = []
    for feat in gj["features"]:
        g = feat["geometry"]
        lineas = [g["coordinates"]] if g["type"] == "LineString" else g["coordinates"]
        for ln in lineas:
            a = np.array(ln)
            dentro = (a[:, 0] > lon_min - 1) & (a[:, 0] < lon_max + 1) & (a[:, 1] > lat_min - 1) & (a[:, 1] < lat_max + 1)
            if dentro.sum() > 1:
                tramos.append(a[dentro])
    return tramos
print("Entorno listo.")

## Paso 1 ✏️ · La ficha de la fuente

Reemplacen **todos** los valores por los de la ficha de su informe (sección 5.10). Respeten la convención de la ficha:

- `lat`, `lon`: proyección en superficie del **centroide** de la falla.
- `z_sup`: profundidad del **borde superior** (km, positiva hacia abajo).
- `rumbo`, `buz`, `rake` en grados, con la regla de la mano derecha (mirando en la dirección del rumbo, la falla buza hacia la derecha).
- `L`, `W` en km; `D` en m; `mu` en GPa.
- `D = None` calcula el deslizamiento medio a partir de Mw: D = M0 / (μ·L·W).

> ⚠️ **Los valores que vienen escritos son un ejemplo ilustrativo** para que el cuaderno funcione desde el primer momento. Rumbo y buzamiento toman los del informe del Servicio Geológico Colombiano para Pedernales 2016; ubicación, profundidad y rake son aproximados, y L y W salen de la escala de Strasser et al. (2010). **No los usen como resultado.**

**Carga desde el CSV del Capítulo 1 (recomendado).** Suban el archivo `fichas_<grupo>.csv` que produce el cuaderno `Cap1_fuente_sismica.ipynb` (ícono de carpeta de la izquierda), escriban su nombre en `FICHA_CSV` y elijan el evento en el menú `EVENTO_CSV`. La celda reemplaza la ficha de ejemplo por la suya, incluidos el **M0 adoptado** y **la localidad del grupo con sus coordenadas**, para que mapa, perfil y sensibilidad salgan de los mismos valores del capítulo.

Sin CSV, la ficha se escribe a mano como antes; para una localidad que no esté en la lista, escriban su nombre en `LOCALIDAD` y sus coordenadas en `LOCALIDAD_PROPIA = (lat, lon)`.

In [ ]:
#@title ✏️ Datos del grupo y ficha de la fuente
# ✏️ Datos del grupo
GRUPO = "Grupo X"  #@param {type:"string"}
LOCALIDAD = "Pedernales"   # nombre de la localidad del perfil (si cargan el CSV, se toma del CSV)
LOCALIDAD_PROPIA = None    # ✏️ (lat, lon) si su localidad no está en la lista; p. ej. (-35.33, -72.41)

#@markdown ✏️ **Recomendado: cargar la ficha desde el CSV del Capítulo 1** (súbanlo con el ícono de carpeta). Con esto se completan solas la ficha y la localidad.
FICHA_CSV = ""  #@param {type:"string"}
EVENTO_CSV = "Pedernales 2016"  #@param ["Pedernales 2016", "Maule 2010", "Illapel 2015", "Iquique 2014", "Pisco 2007", "Camaná 2001", "Tohoku 2011", "Nicaragua 1992", "Mentawai 2010", "Samoa 2009", "Océano Índico 2012"]

# ✏️ Ficha de la fuente (EJEMPLO ILUSTRATIVO: reemplacen con su ficha)
FICHA = dict(
    evento = "Pedernales 2016",
    Mw     = 7.8,
    lat    = 0.10,     # centroide, grados
    lon    = -80.25,   # centroide, grados
    z_sup  = 10.0,     # km
    rumbo  = 29.0,     # grados
    buz    = 32.0,     # grados
    rake   = 90.0,     # grados
    L      = 122.0,    # km
    W      = 72.0,     # km
    D      = None,     # m (None = calcular desde Mw)
    mu     = 30.0,     # GPa
)
NU = 0.25   # coeficiente de Poisson del semiespacio (λ = μ)

if LOCALIDAD_PROPIA:
    LOCALIDADES[LOCALIDAD] = tuple(LOCALIDAD_PROPIA)

if FICHA_CSV:
    _f = pd.read_csv(FICHA_CSV)
    _f = _f[_f["evento"] == EVENTO_CSV] if EVENTO_CSV else _f
    if _f.empty:
        raise ValueError(f"El CSV no tiene filas del evento '{EVENTO_CSV}'. Eventos disponibles: {sorted(pd.read_csv(FICHA_CSV)['evento'].unique())}")
    if _f["evento"].nunique() > 1:
        raise ValueError("El CSV tiene varios eventos: indiquen cuál en EVENTO_CSV.")
    _v = dict(zip(_f["parametro"], _f["valor"].astype(float)))
    FICHA = dict(evento=_f["evento"].iloc[0], Mw=_v.get("Mw"), M0=_v.get("M0"), lat=_v["lat"], lon=_v["lon"],
                 z_sup=_v["z_sup"], rumbo=_v["rumbo"], buz=_v["buz"], rake=_v["rake"], L=_v["L"], W=_v["W"],
                 D=_v.get("D"), mu=_v["mu"])
    if "loc_lat" in _v and "loc_lon" in _v:
        LOCALIDAD = str(_f.loc[_f["parametro"] == "loc_lat", "metodo"].iloc[0]) if "metodo" in _f else "Localidad del grupo"
        LOCALIDADES[LOCALIDAD] = (_v["loc_lat"], _v["loc_lon"])
        print(f"Localidad tomada del CSV: {LOCALIDAD} ({_v['loc_lat']:.3f}°, {_v['loc_lon']:.3f}°)")
    if FICHA["Mw"] is None and FICHA["M0"]:
        FICHA["Mw"] = (math.log10(FICHA["M0"]) - 9.1) / 1.5
    print(f"Ficha cargada desde {FICHA_CSV}: {FICHA['evento']}")
    display(_f[["parametro", "valor", "unidad", "fuente"]])

## Paso 2 · Auditoría de la ficha

Ningún modelo es mejor que sus datos de entrada. Esta celda revisa la coherencia interna de la ficha. **Si aparece un ❌, corrijan la ficha antes de seguir.** Las ⚠️ no detienen el trabajo, pero deben comentarse en el informe.

In [ ]:
F = dict(FICHA)
M0 = F.get("M0") or 10**(1.5 * F["Mw"] + 9.1)       # N·m: el de la ficha o, si no hay, desde Mw (Hanks y Kanamori, 1979)
mu_Pa = F["mu"] * 1e9
if F["D"] is None:
    F["D"] = M0 / (mu_Pa * F["L"] * 1e3 * F["W"] * 1e3)
    origen_D = "calculado desde Mw"
else:
    origen_D = "dado en la ficha"

dip = math.radians(F["buz"])
z_inf = F["z_sup"] + F["W"] * math.sin(dip)
z_cen = F["z_sup"] + F["W"] / 2 * math.sin(dip)
M0_ficha = mu_Pa * F["L"] * 1e3 * F["W"] * 1e3 * F["D"]
Mw_ficha = (math.log10(M0_ficha) - 9.1) / 1.5
L_str = 10**(-2.477 + 0.585 * F["Mw"])               # Strasser et al. (2010), interplaca
W_str = 10**(-0.882 + 0.351 * F["Mw"])

filas = []
def chequeo(nombre, valor, estado, comentario):
    filas.append({"Verificación": nombre, "Valor": valor, "Estado": estado, "Comentario": comentario})

vacios = [k for k, v in FICHA.items() if v is None and k not in ("D", "M0")]
chequeo("Celdas completas", "sí" if not vacios else ", ".join(vacios), "✅" if not vacios else "❌",
        "La ficha no puede tener celdas vacías")
chequeo("Localidad reconocida", LOCALIDAD, "✅" if LOCALIDAD in LOCALIDADES else "❌",
        "Escríbanla como en la lista o definan LOCALIDAD_PROPIA = (lat, lon)")
chequeo("Profundidad del borde superior", f"{F['z_sup']:.1f} km",
        "❌" if F["z_sup"] < 0 else ("⚠️" if F["z_sup"] < 1 else "✅"),
        "z_sup < 0: la falla sale por encima del fondo; z_sup < 1 km: solución casi singular en la traza")
chequeo("Buzamiento", f"{F['buz']:.1f}°",
        "❌" if not 0 < F["buz"] < 90 else ("✅" if 5 <= F["buz"] <= 35 else "⚠️"),
        "Interplaca típico: 5°–35°")
r = ((F["rake"] + 180) % 360) - 180
tipo = ("inversa" if 45 <= r <= 135 else "normal" if -135 <= r <= -45 else "de rumbo / oblicua")
chequeo("Rake", f"{F['rake']:.1f}° → falla {tipo}", "✅" if tipo == "inversa" else "⚠️",
        "Un sismo de subducción interplaca debería ser inverso")
razon = M0_ficha / M0
chequeo("Momento: μ·L·W·D frente a Mw", f"{razon:.2f} (Mw implícito {Mw_ficha:.2f})",
        "✅" if 0.8 <= razon <= 1.25 else "⚠️", "Cociente aceptable entre 0,8 y 1,25")
chequeo("Largo frente a Strasser", f"L = {F['L']:.0f} km (escala: {L_str:.0f} km)",
        "✅" if 0.5 <= F["L"] / L_str <= 2 else "⚠️", "Diferencias mayores a ×2 deben justificarse")
chequeo("Ancho frente a Strasser", f"W = {F['W']:.0f} km (escala: {W_str:.0f} km)",
        "✅" if 0.5 <= F["W"] / W_str <= 2 else "⚠️", "Diferencias mayores a ×2 deben justificarse")
chequeo("Profundidad del borde inferior", f"{z_inf:.1f} km", "✅" if z_inf <= 55 else "⚠️",
        "Más allá de ~50–55 km la interfaz suele dejar de ser sismogénica")

U1, U2 = F["D"] * math.cos(math.radians(F["rake"])), F["D"] * math.sin(math.radians(F["rake"]))
display(pd.DataFrame(filas))
print(f"\nDeslizamiento medio D = {F['D']:.2f} m ({origen_D});  U1 (rumbo) = {U1:.2f} m;  U2 (buzamiento) = {U2:.2f} m")
print(f"Profundidad del centroide = {z_cen:.1f} km;  borde inferior = {z_inf:.1f} km")
print(f"Salto vertical entre bloques en la falla, D·sen δ = {F['D']*math.sin(dip):.2f} m  (NO es el techo del levantamiento en superficie)")
if any(f["Estado"] == "❌" for f in filas):
    raise ValueError("La ficha tiene errores (❌). Corríjanla antes de continuar.")

## Paso 3 · ¿El código hace lo que dice?

Antes de confiar en un modelo, se verifica. Okada (1985, tabla 2) publicó valores de control para un caso de prueba (x = 2, y = 3, d = 4, δ = 70°, L = 3, W = 2, λ = μ). Si el código está bien, debe reproducirlos.

El código implementa la solución de Okada (1985) para los desplazamientos en superficie de una dislocación rectangular en un semiespacio elástico homogéneo. Pueden leerlo, pero no lo modifiquen.

In [ ]:
import numpy as np

EPS = 1e-14

def _I5(xi, eta, q, dip, nu, R, db):
    X = np.sqrt(xi**2 + q**2)
    cd, sd = np.cos(dip), np.sin(dip)
    if cd > EPS:
        with np.errstate(divide='ignore', invalid='ignore'):
            I = (1 - 2*nu) * 2/cd * np.arctan((eta*(X + q*cd) + X*(R + X)*sd) / (xi*(R + X)*cd))
        I = np.where(xi == 0, 0.0, I)
    else:
        I = -(1 - 2*nu) * xi*sd/(R + db)
    return I

def _I4(db, eta, q, dip, nu, R):
    cd, sd = np.cos(dip), np.sin(dip)
    if cd > EPS:
        return (1 - 2*nu) / cd * (np.log(R + db) - sd*np.log(R + eta))
    return -(1 - 2*nu) * q/(R + db)

def _I3(eta, q, dip, nu, R):
    cd, sd = np.cos(dip), np.sin(dip)
    yb = eta*cd + q*sd
    db = eta*sd - q*cd
    if cd > EPS:
        return (1 - 2*nu) * (yb/(cd*(R + db)) - np.log(R + eta)) + sd/cd*_I4(db, eta, q, dip, nu, R)
    return (1 - 2*nu)/2 * (eta/(R + db) + yb*q/(R + db)**2 - np.log(R + eta))

def _I2(eta, q, dip, nu, R):
    return (1 - 2*nu) * (-np.log(R + eta)) - _I3(eta, q, dip, nu, R)

def _I1(xi, eta, q, dip, nu, R):
    cd, sd = np.cos(dip), np.sin(dip)
    db = eta*sd - q*cd
    if cd > EPS:
        return (1 - 2*nu) * (-xi/(cd*(R + db))) - sd/cd*_I5(xi, eta, q, dip, nu, R, db)
    return -(1 - 2*nu)/2 * xi*q/(R + db)**2

def _atan_term(xi, eta, q, R):
    with np.errstate(divide='ignore', invalid='ignore'):
        t = np.arctan(xi*eta/(q*R))
    return np.where(q != 0, t, 0.0)

def _ux_ss(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    return xi*q/(R*(R + eta)) + _atan_term(xi, eta, q, R) + _I1(xi, eta, q, dip, nu, R)*np.sin(dip)

def _uy_ss(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    cd, sd = np.cos(dip), np.sin(dip)
    return (eta*cd + q*sd)*q/(R*(R + eta)) + q*cd/(R + eta) + _I2(eta, q, dip, nu, R)*sd

def _uz_ss(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    cd, sd = np.cos(dip), np.sin(dip)
    db = eta*sd - q*cd
    return db*q/(R*(R + eta)) + q*sd/(R + eta) + _I4(db, eta, q, dip, nu, R)*sd

def _ux_ds(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    return q/R - _I3(eta, q, dip, nu, R)*np.sin(dip)*np.cos(dip)

def _uy_ds(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    cd, sd = np.cos(dip), np.sin(dip)
    return ((eta*cd + q*sd)*q/(R*(R + xi)) + cd*_atan_term(xi, eta, q, R)
            - _I1(xi, eta, q, dip, nu, R)*sd*cd)

def _uz_ds(xi, eta, q, dip, nu):
    R = np.sqrt(xi**2 + eta**2 + q**2)
    cd, sd = np.cos(dip), np.sin(dip)
    db = eta*sd - q*cd
    return (db*q/(R*(R + xi)) + sd*_atan_term(xi, eta, q, R)
            - _I5(xi, eta, q, dip, nu, R, db)*sd*cd)

def _chinnery(f, x, p, L, W, q, dip, nu):
    return f(x, p, q, dip, nu) - f(x, p - W, q, dip, nu) - f(x - L, p, q, dip, nu) + f(x - L, p - W, q, dip, nu)

def okada_marco_falla(x, y, d, dip_deg, L, W, U1, U2, nu=0.25):
    """Desplazamientos (ux, uy, uz) en el sistema de Okada (1985).
    Origen en el extremo del borde INFERIOR; x a lo largo del rumbo; d = profundidad del borde inferior."""
    dip = np.radians(dip_deg)
    x = np.asarray(x, float); y = np.asarray(y, float)
    p = y*np.cos(dip) + d*np.sin(dip)
    q = y*np.sin(dip) - d*np.cos(dip)
    ux = -U1/(2*np.pi)*_chinnery(_ux_ss, x, p, L, W, q, dip, nu) - U2/(2*np.pi)*_chinnery(_ux_ds, x, p, L, W, q, dip, nu)
    uy = -U1/(2*np.pi)*_chinnery(_uy_ss, x, p, L, W, q, dip, nu) - U2/(2*np.pi)*_chinnery(_uy_ds, x, p, L, W, q, dip, nu)
    uz = -U1/(2*np.pi)*_chinnery(_uz_ss, x, p, L, W, q, dip, nu) - U2/(2*np.pi)*_chinnery(_uz_ds, x, p, L, W, q, dip, nu)
    return ux, uy, uz

def okada_centroide(e, n, prof_centroide, rumbo, buz, L, W, rake, D, nu=0.25):
    """Desplazamientos superficiales (uE, uN, uZ) para una falla rectangular.
    e, n: coordenadas Este/Norte relativas a la proyección superficial del CENTROIDE (misma unidad que L, W).
    prof_centroide: profundidad del centroide (positiva hacia abajo)."""
    s, dp = np.radians(rumbo), np.radians(buz)
    U1 = D*np.cos(np.radians(rake)); U2 = D*np.sin(np.radians(rake))
    d = prof_centroide + np.sin(dp)*W/2            # profundidad del borde inferior
    ec = e + np.cos(s)*np.cos(dp)*W/2
    nc = n - np.sin(s)*np.cos(dp)*W/2
    x = np.cos(s)*nc + np.sin(s)*ec + L/2
    y = np.sin(s)*nc - np.cos(s)*ec + np.cos(dp)*W
    ux, uy, uz = okada_marco_falla(x, y, d, buz, L, W, U1, U2, nu)
    uE = np.sin(s)*ux - np.cos(s)*uy
    uN = np.cos(s)*ux + np.sin(s)*uy
    return uE, uN, uz


# --- Verificación contra Okada (1985), tabla 2, caso 2 ---
ref = {"rumbo (U1=1)": (-8.689e-3, -4.298e-3, -2.747e-3),
       "buzamiento (U2=1)": (-4.682e-3, -3.527e-2, -3.564e-2)}
calc = {"rumbo (U1=1)": okada_marco_falla(2, 3, 4, 70, 3, 2, 1, 0),
        "buzamiento (U2=1)": okada_marco_falla(2, 3, 4, 70, 3, 2, 0, 1)}
tabla = []
for k in ref:
    for comp, r_, c_ in zip(("ux", "uy", "uz"), ref[k], calc[k]):
        tabla.append({"Caso": k, "Componente": comp, "Okada (1985)": r_, "Este código": round(float(c_), 6),
                      "Coincide": "✅" if abs(c_ - r_) < 1e-6 else "❌"})
display(pd.DataFrame(tabla))

## Paso 4 · El campo de deformación

Se calcula el desplazamiento en una malla alrededor de la falla. En el mapa:

- **Color**: desplazamiento vertical Uz (rojo sube, azul baja).
- **Línea negra discontinua**: línea de cero, donde se pasa de levantamiento a hundimiento.
- **Rectángulo**: proyección de la falla; el lado grueso es el **borde superior**.
- **Flechas**: desplazamiento horizontal.

Recuerden la convención: el sistema de Okada usa z positivo hacia arriba, así que Uz > 0 es levantamiento.

In [ ]:
lat0, lon0 = F["lat"], F["lon"]
s, d_ = math.radians(F["rumbo"]), dip

def esquinas_falla():
    # Esquinas de la proyección de la falla en superficie (km respecto al centroide)
    us = np.array([math.sin(s), math.cos(s)])                     # vector a lo largo del rumbo (E, N)
    ub = np.array([math.sin(s + math.pi/2), math.cos(s + math.pi/2)])  # dirección del buzamiento
    h = F["W"] / 2 * math.cos(d_)
    sup = [-h * ub - F["L"]/2 * us, -h * ub + F["L"]/2 * us]
    inf = [ h * ub + F["L"]/2 * us,  h * ub - F["L"]/2 * us]
    return np.array(sup), np.array(inf)

def campo(Fp, e, n):
    zc = Fp["z_sup"] + Fp["W"] / 2 * math.sin(math.radians(Fp["buz"]))
    uE, uN, uZ = okada_centroide(e, n, zc, Fp["rumbo"], Fp["buz"], Fp["L"], Fp["W"], Fp["rake"], Fp["D"], NU)
    return uE, uN, uZ

ext = F["L"] / 2 + F["W"] + 120
paso = 2.0
eg, ng = np.meshgrid(np.arange(-ext, ext + paso, paso), np.arange(-ext, ext + paso, paso))
uE, uN, uZ = campo(F, eg, ng)
LATg, LONg = a_grados(eg, ng, lat0, lon0)

iz_max, iz_min = np.unravel_index(np.nanargmax(uZ), uZ.shape), np.unravel_index(np.nanargmin(uZ), uZ.shape)
uh = np.hypot(uE, uN)
print(f"Levantamiento máximo: {uZ[iz_max]:.2f} m en ({LATg[iz_max]:.2f}°, {LONg[iz_max]:.2f}°)")
print(f"Hundimiento máximo:  {uZ[iz_min]:.2f} m en ({LATg[iz_min]:.2f}°, {LONg[iz_min]:.2f}°)")
print(f"Desplazamiento horizontal máximo: {np.nanmax(uh):.2f} m")

bbox = (LONg.min(), LONg.max(), LATg.min(), LATg.max())
COSTA = cargar_costa(bbox)

def mapa(ax, Z, titulo, cbar_label="Uz (m)"):
    lim = np.nanmax(np.abs(Z))
    pm = ax.pcolormesh(LONg, LATg, Z, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), shading="auto")
    cs = ax.contour(LONg, LATg, Z, levels=[0], colors="k", linestyles="--", linewidths=1.2)
    niveles = [v for v in np.round(np.linspace(-lim, lim, 9), 2) if abs(v) > 1e-3]
    c2 = ax.contour(LONg, LATg, Z, levels=niveles, colors="k", linewidths=0.4, alpha=0.6)
    ax.clabel(c2, fmt="%.2f", fontsize=7)
    for t in COSTA:
        ax.plot(t[:, 0], t[:, 1], color="0.15", lw=0.9)
    sup, inf = esquinas_falla()
    poli = np.vstack([sup, inf, sup[:1]])
    la, lo = a_grados(poli[:, 0], poli[:, 1], lat0, lon0)
    ax.plot(lo, la, color="0.2", lw=1)
    la, lo = a_grados(sup[:, 0], sup[:, 1], lat0, lon0)
    ax.plot(lo, la, color="0.2", lw=3.5, label="borde superior")
    for nombre, (la_, lo_) in LOCALIDADES.items():
        es = nombre == LOCALIDAD
        ax.plot(lo_, la_, marker="*" if es else "o", ms=14 if es else 4, color="gold" if es else "k",
                mec="k", zorder=5)
        ax.annotate(nombre, (lo_, la_), xytext=(4, 3), textcoords="offset points", fontsize=8,
                    fontweight="bold" if es else "normal")
    ax.set_xlim(bbox[0], bbox[1]); ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect(1 / math.cos(math.radians(lat0)))
    ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)"); ax.set_title(titulo)
    plt.colorbar(pm, ax=ax, shrink=0.8, label=cbar_label)
    return pm

fig, ax = plt.subplots(figsize=(8, 8))
mapa(ax, uZ, f"{GRUPO} · {FICHA['evento']}\nDesplazamiento vertical (Okada, 1985)")
k = max(1, int(25 / paso))
esc = np.nanmax(uh) * 12
ax.quiver(LONg[::k, ::k], LATg[::k, ::k], uE[::k, ::k], uN[::k, ::k], scale=esc, width=0.003, color="0.1")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.savefig("fig1_mapa_uz.png", dpi=200); plt.show()

**Para leer el mapa (respondan en el informe):** ¿El levantamiento máximo queda sobre el mar o sobre tierra? ¿Hacia dónde se mueve horizontalmente la costa? ¿Por qué el levantamiento máximo puede ser mayor que D·sen δ?

## Paso 5 · Perfil por su localidad

El perfil corta la falla **perpendicular al rumbo** y pasa por la localidad del grupo. La distancia es negativa hacia el mar (fosa) y positiva hacia tierra. El panel inferior muestra el corte de la falla en profundidad.

Si la localidad queda fuera del largo L de la falla, el perfil no la corta en su parte central: lo verán porque el desplazamiento se atenúa.

In [ ]:
lat_loc, lon_loc = LOCALIDADES[LOCALIDAD]
e_loc, n_loc = a_km(lat_loc, lon_loc, lat0, lon0)
az_tierra = s + math.pi / 2                          # dirección del buzamiento (hacia tierra)
ub = np.array([math.sin(az_tierra), math.cos(az_tierra)])
us = np.array([math.sin(s), math.cos(s)])
dist = np.arange(-250, 150.5, 0.5)
ep, np_ = e_loc + dist * ub[0], n_loc + dist * ub[1]
pE, pN, pZ = campo(F, ep, np_)
pH = pE * ub[0] + pN * ub[1]                          # componente horizontal a lo largo del perfil (+ hacia tierra)

# Posición de los bordes de la falla en la coordenada del perfil
pos_loc = e_loc * ub[0] + n_loc * ub[1]               # posición de la localidad en dirección del buzamiento
h = F["W"] / 2 * math.cos(dip)
s_sup, s_inf = -h - pos_loc, h - pos_loc
fuera = abs(e_loc * us[0] + n_loc * us[1]) > F["L"] / 2

uE0, uN0, uZ0 = campo(F, np.array(e_loc), np.array(n_loc))
LOC = dict(uz=float(uZ0), uh=float(np.hypot(uE0, uN0)),
           az=float((math.degrees(math.atan2(uE0, uN0)) + 360) % 360))

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(9, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1.4, 1.4]})
a1.axhline(0, color="k", lw=0.8)
a1.plot(dist, pZ, color="C3", lw=2, label="Uz (modelo)")
a1.axhline(F["D"] * math.sin(dip), color="0.4", ls=":", label="D·sen δ (salto en la falla)")
a1.axvline(0, color="goldenrod", lw=2, label=LOCALIDAD)
a1.set_ylabel("Uz (m)"); a1.legend(fontsize=8); a1.set_title(f"Perfil perpendicular al rumbo por {LOCALIDAD}")
a2.axhline(0, color="k", lw=0.8)
a2.plot(dist, pH, color="C0", lw=2)
a2.axvline(0, color="goldenrod", lw=2)
a2.set_ylabel("U horizontal (m)\n(+ hacia tierra)")
a3.plot([s_sup, s_inf], [-F["z_sup"], -z_inf], color="k", lw=3)
a3.annotate("borde superior", (s_sup, -F["z_sup"]), xytext=(-60, 8), textcoords="offset points", fontsize=8)
a3.axvline(0, color="goldenrod", lw=2)
a3.axhline(0, color="0.5", lw=0.8)
a3.set_ylim(-max(60, z_inf + 10), 5); a3.set_ylabel("Profundidad (km)")
a3.set_xlabel("Distancia a la localidad (km)   ← mar · tierra →")
plt.tight_layout(); plt.savefig("fig2_perfil.png", dpi=200); plt.show()

print(f"En {LOCALIDAD}: Uz = {LOC['uz']:+.2f} m;  horizontal = {LOC['uh']:.2f} m hacia el azimut {LOC['az']:.0f}°")
if fuera:
    print("⚠️ La localidad queda fuera del largo L de la falla: el perfil pasa por un extremo o fuera de ella.")

## Paso 6 · Predicción frente a modelo

Comparen lo que predijeron en el Paso 0 con lo que calculó el modelo. La pregunta clave para el informe no es cuánto se equivocaron, sino **qué razonamiento falló o acertó**.

In [ ]:
signo_modelo = "sube" if LOC["uz"] > 0 else "baja"
comp = pd.DataFrame([
    {"Magnitud": "Levantamiento máximo (m)", "Predicción": PREDICCION["levantamiento_max_m"], "Modelo": round(float(uZ[iz_max]), 2)},
    {"Magnitud": "Hundimiento máximo (m)", "Predicción": PREDICCION["hundimiento_max_m"], "Modelo": round(float(uZ[iz_min]), 2)},
    {"Magnitud": f"Signo en {LOCALIDAD}", "Predicción": PREDICCION["signo_en_localidad"], "Modelo": signo_modelo},
])
display(comp)
print("Justificación registrada:", PREDICCION["justificacion"])

## Paso 7 ✏️ · Validación con observaciones

Ingresen las observaciones de deformación que reunieron en la sección 5.9 del informe (GPS, InSAR, marcas biológicas o mediciones de campo). Cada observación necesita coordenadas, valor y fuente. Si su evento no tiene datos verticales, usen los horizontales (`ue_obs_m`, `un_obs_m`) o expliquen en el informe por qué no se puede validar.

Copien la línea de ejemplo tantas veces como observaciones tengan y **borren el ejemplo**.

In [ ]:
# ✏️ Observaciones (EJEMPLO: borren esta línea y escriban las suyas)
OBSERVACIONES = [
    # dict(sitio="EJEMPLO", lat=0.07, lon=-80.05, uz_obs_m=-0.30, ue_obs_m=None, un_obs_m=None, fuente="Autor (año)"),
]

In [ ]:
if not OBSERVACIONES:
    print("Sin observaciones cargadas. Completen la celda anterior o justifiquen en el informe por qué no es posible validar.")
else:
    obs = pd.DataFrame(OBSERVACIONES)
    eo, no = a_km(obs["lat"].values, obs["lon"].values, lat0, lon0)
    mE, mN, mZ = campo(F, eo, no)
    obs["uz_mod_m"], obs["ue_mod_m"], obs["un_mod_m"] = mZ.round(3), mE.round(3), mN.round(3)
    pares = []
    for c in ("uz", "ue", "un"):
        if f"{c}_obs_m" in obs:
            v = obs[f"{c}_obs_m"].astype(float)
            obs[f"{c}_res_m"] = (v - obs[f"{c}_mod_m"]).round(3)
            ok = v.notna()
            if ok.any():
                pares.append((c, v[ok].values, obs.loc[ok, f"{c}_mod_m"].values))
    display(obs)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    todos = []
    for c, o, m in pares:
        ax.scatter(o, m, label=f"{c}  (RMSE = {np.sqrt(np.mean((o - m)**2)):.2f} m)", s=40, edgecolor="k")
        todos += list(o) + list(m)
    lim = max(abs(np.array(todos))) * 1.15
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xlabel("Observado (m)"); ax.set_ylabel("Modelo (m)"); ax.legend(fontsize=8)
    ax.set_title("Validación: observado frente a modelo")
    plt.tight_layout(); plt.savefig("fig3_validacion.png", dpi=200); plt.show()

## Paso 8 ✏️ · Sensibilidad: ¿qué parámetro manda?

Se cambia **un parámetro a la vez** dentro de su rango de incertidumbre y se mide cuánto cambian tres resultados: el levantamiento máximo, el hundimiento máximo y Uz en su localidad.

Decisión importante: con `CONSERVAR_M0 = True`, al cambiar L, W o μ se recalcula D para que el sismo siga teniendo la misma magnitud. Es lo físicamente coherente: no se puede agrandar la falla sin repartir el mismo momento. Prueben también con `False` y comenten la diferencia.

Ajusten los rangos con la incertidumbre que discutieron en la sección 5.11 de su informe.

In [ ]:
# ✏️ Rangos de incertidumbre (valor bajo, valor alto)
RANGOS = {
    "z_sup": (max(0.5, F["z_sup"] * 0.5), F["z_sup"] * 1.5),
    "buz":   (F["buz"] - 5, F["buz"] + 5),
    "rake":  (F["rake"] - 15, F["rake"] + 15),
    "L":     (F["L"] * 0.8, F["L"] * 1.2),
    "W":     (F["W"] * 0.8, F["W"] * 1.2),
    "mu":    (30.0, 40.0),
}
CONSERVAR_M0 = True

In [ ]:
paso_s = 4.0
es_, ns_ = np.meshgrid(np.arange(-ext, ext + paso_s, paso_s), np.arange(-ext, ext + paso_s, paso_s))

def metricas(Fp):
    if CONSERVAR_M0:
        Fp = dict(Fp, D=M0 / (Fp["mu"] * 1e9 * Fp["L"] * 1e3 * Fp["W"] * 1e3))
    _, _, z = campo(Fp, es_, ns_)
    _, _, zl = campo(Fp, np.array(e_loc), np.array(n_loc))
    return {"Levantamiento máx.": np.nanmax(z), "Hundimiento máx.": np.nanmin(z), f"Uz en {LOCALIDAD}": float(zl)}

base = metricas(F)
res = []
for p, (lo_, hi_) in RANGOS.items():
    m_lo, m_hi = metricas(dict(F, **{p: lo_})), metricas(dict(F, **{p: hi_}))
    for mt in base:
        res.append({"Parámetro": p, "Rango": f"{lo_:.1f} – {hi_:.1f}", "Métrica": mt,
                    "Base": base[mt], "Con valor bajo": m_lo[mt], "Con valor alto": m_hi[mt]})
sens = pd.DataFrame(res)
display(sens.round(3))

metr = list(base)
fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, mt in zip(axs, metr):
    sub = sens[sens["Métrica"] == mt].copy()
    sub["amp"] = (sub["Con valor alto"] - sub["Con valor bajo"]).abs()
    sub = sub.sort_values("amp")
    y = np.arange(len(sub))
    ax.barh(y, sub["Con valor bajo"] - sub["Base"], color="C0", label="valor bajo")
    ax.barh(y, sub["Con valor alto"] - sub["Base"], color="C3", alpha=0.75, label="valor alto")
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(sub["Parámetro"])
    ax.set_title(f"{mt}\n(base = {base[mt]:.2f} m)", fontsize=9); ax.set_xlabel("Cambio respecto a la base (m)")
axs[0].legend(fontsize=8)
plt.suptitle(f"Sensibilidad · conservar M0 = {CONSERVAR_M0}", fontsize=10)
plt.tight_layout(); plt.savefig("fig4_sensibilidad.png", dpi=200); plt.show()

## Paso 9 · Condición inicial para la Unidad 3

En la Unidad 3 se propagará el tsunami. La aproximación más usada toma como superficie inicial del mar el desplazamiento vertical del fondo marino (Uz), porque la ruptura es mucho más rápida que la respuesta del océano. Esta celda guarda los archivos que necesitarán:

- `condicion_inicial_<grupo>.csv`: lon, lat, uE, uN, uZ en la malla.
- `ficha_final_<grupo>.json`: ficha usada, D final y resultados clave.
- Las cuatro figuras PNG para el informe.

In [ ]:
nombre = GRUPO.replace(" ", "_")
pd.DataFrame({"lon": LONg.ravel(), "lat": LATg.ravel(), "uE_m": uE.ravel(), "uN_m": uN.ravel(),
              "uZ_m": uZ.ravel()}).round(5).to_csv(f"condicion_inicial_{nombre}.csv", index=False)
resumen = dict(grupo=GRUPO, localidad=LOCALIDAD, ficha=F, nu=NU, M0_Nm=M0,
               z_centroide_km=z_cen, z_inferior_km=z_inf,
               levantamiento_max_m=float(uZ[iz_max]), hundimiento_max_m=float(uZ[iz_min]),
               en_localidad=LOC, prediccion=PREDICCION)
with open(f"ficha_final_{nombre}.json", "w") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2, default=float)
archivos = [f"condicion_inicial_{nombre}.csv", f"ficha_final_{nombre}.json",
            "fig1_mapa_uz.png", "fig2_perfil.png", "fig3_validacion.png", "fig4_sensibilidad.png"]
archivos = [a for a in archivos if os.path.exists(a)]
print("Archivos generados:", *archivos, sep="\n  ")
try:
    from google.colab import files
    for a in archivos:
        files.download(a)
except ImportError:
    pass

## Preguntas para el informe

1. **Predicción.** ¿Qué acertaron y qué no en el Paso 0? ¿Qué idea previa corrigieron?
2. **Patrón.** Describan el campo de deformación: zona de levantamiento, zona de hundimiento, línea de cero y movimiento horizontal. Relaciónenlo con la geometría de la falla.
3. **Más allá de D·sen δ.** Expliquen con su resultado por qué el levantamiento en superficie puede superar el salto vertical en la falla.
4. **Su localidad.** ¿Sube o baja? ¿Cuánto? ¿Qué consecuencias tiene para la inundación por tsunami, la intrusión marina, el drenaje y la infraestructura portuaria?
5. **Validación.** ¿Dónde acierta el modelo y dónde no? Propongan causas: deslizamiento uniforme frente a heterogéneo, semiespacio homogéneo, deformación postsísmica incluida en los datos, ubicación del centroide.
6. **Sensibilidad.** ¿Qué parámetro controla más el resultado en su localidad? ¿Coincide con el que consideraron más incierto en la Unidad 1? ¿Qué dato adicional reduciría más la incertidumbre?
7. **Pedernales como referencia.** Comparen su evento con el caso común: tamaño, profundidad, patrón de deformación.

**Referencias del cuaderno**
- Hanks, T. C., y Kanamori, H. (1979). A moment magnitude scale. *Journal of Geophysical Research, 84*(B5), 2348–2350.
- Okada, Y. (1985). Surface deformation due to shear and tensile faults in a half-space. *Bulletin of the Seismological Society of America, 75*(4), 1135–1154.
- Strasser, F. O., Arango, M. C., y Bommer, J. J. (2010). Scaling of the source dimensions of interface and intraslab subduction-zone earthquakes with moment magnitude. *Seismological Research Letters, 81*(6), 941–950.
- Línea de costa: Natural Earth (dominio público).